# RappiPay Fraud Detection Lab
## Construye un Cortex Agent con Cortex Code

**Autor**: Juan Camilo Villarreal | **Duracion**: 45 minutos

---

### Objetivo
Construir un agente AI conversacional (Cortex Agent) que responda preguntas sobre fraude de RappiPay en lenguaje natural.

### Que vas a aprender
1. Configurar un ambiente de datos de fraude en Snowflake
2. Usar **Cortex Code** para explorar datos y generar SQL
3. Crear una **Semantic View** para ensenarle a la AI tu modelo de datos
4. Crear un **Cortex Agent** y habilitarlo en **Snowflake Intelligence**

### Prerequisitos
- Cuenta Snowflake ([Solicita aqui](https://go.dataops.live/rappy-day/instructions))
- Cortex Code CLI instalado (ver instrucciones abajo)

### Instalar Cortex Code CLI

**macOS y Linux (incluyendo WSL):**
```bash
curl -LsS https://ai.snowflake.com/static/cc-scripts/install.sh | sh
```

**Windows (PowerShell):**
```powershell
irm https://ai.snowflake.com/static/cc-scripts/install.ps1 | iex
```

**Desktop App** (alternativa): [Descargar aqui](https://www.snowflake.com/en/product/snowflake-coco/downloads/)

### Contexto
Eres un Data Engineer en RappiPay. Tu equipo de fraude necesita consultar alertas y metricas sin escribir SQL. Vas a construir un asistente AI que responda sus preguntas en espanol.

> **Nota**: Los datos son ficticios pero inspirados en el mercado fintech colombiano/mexicano.
> **Tip permisos**: Cuando Cortex Code pida confirmacion de permisos SQL, selecciona "Allow any statement in RAPPIPAY_DB" para continuar sin interrupciones.

---
## Task 1: Setup del Ambiente (10 min)

**Objetivo**: Crear la base de datos, tablas y cargar datos sinteticos de fraude.

### Paso 1.1: Crear Database y Schemas

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE DATABASE RAPPIPAY_DB;

CREATE OR REPLACE WAREHOUSE RAPPIPAY_WH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE RAPPIPAY_WH;
USE DATABASE RAPPIPAY_DB;

CREATE OR REPLACE SCHEMA RAPPIPAY_DB.RAW;
CREATE OR REPLACE SCHEMA RAPPIPAY_DB.CURATED;
CREATE OR REPLACE SCHEMA RAPPIPAY_DB.ANALYTICS;
CREATE OR REPLACE SCHEMA RAPPIPAY_DB.APP;

### Paso 1.2: Crear Tablas

In [ ]:
USE SCHEMA RAPPIPAY_DB.RAW;

CREATE OR REPLACE TABLE RAPPIPAY_DB.RAW.TRANSACTIONS (
    transaction_id VARCHAR(36) NOT NULL,
    user_id VARCHAR(20) NOT NULL,
    merchant_id VARCHAR(20) NOT NULL,
    amount NUMBER(18,2) NOT NULL,
    currency VARCHAR(3) DEFAULT 'COP',
    transaction_type VARCHAR(30),
    channel VARCHAR(20),
    device_type VARCHAR(20),
    ip_address VARCHAR(45),
    location_city VARCHAR(50),
    location_country VARCHAR(3),
    status VARCHAR(30),
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    description VARCHAR(500)
);

CREATE OR REPLACE TABLE RAPPIPAY_DB.RAW.USERS (
    user_id VARCHAR(20) NOT NULL,
    full_name VARCHAR(100) NOT NULL,
    email VARCHAR(150),
    phone VARCHAR(20),
    country VARCHAR(3),
    city VARCHAR(50),
    registration_date DATE,
    kyc_status VARCHAR(20),
    credit_score NUMBER(3),
    risk_level VARCHAR(10),
    verticals_used NUMBER(2),
    total_transactions NUMBER(10),
    account_status VARCHAR(20)
);

CREATE OR REPLACE TABLE RAPPIPAY_DB.RAW.MERCHANTS (
    merchant_id VARCHAR(20) NOT NULL,
    merchant_name VARCHAR(100) NOT NULL,
    category VARCHAR(50),
    city VARCHAR(50),
    country VARCHAR(3),
    risk_score NUMBER(5,2),
    avg_transaction_amount NUMBER(18,2),
    total_transactions NUMBER(10)
);

CREATE OR REPLACE TABLE RAPPIPAY_DB.RAW.FRAUD_ALERTS (
    alert_id VARCHAR(36) NOT NULL,
    transaction_id VARCHAR(36) NOT NULL,
    user_id VARCHAR(20) NOT NULL,
    alert_type VARCHAR(30),
    severity VARCHAR(10),
    status VARCHAR(30),
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    resolved_at TIMESTAMP_NTZ,
    resolution_notes VARCHAR(1000),
    investigator_notes VARCHAR(2000)
);

### Paso 1.3: Cargar Datos Sinteticos

In [ ]:
-- Merchants (120 registros)
INSERT INTO RAPPIPAY_DB.RAW.MERCHANTS (merchant_id, merchant_name, category, city, country, risk_score, avg_transaction_amount, total_transactions)
SELECT * FROM (
    SELECT 'MCH-' || LPAD(SEQ4()::VARCHAR, 4, '0'),
           CASE MOD(SEQ4(), 40)
               WHEN 0 THEN 'Tienda D1' WHEN 1 THEN 'Exito' WHEN 2 THEN 'Oxxo'
               WHEN 3 THEN 'Rappi Restaurantes' WHEN 4 THEN 'Alkosto' WHEN 5 THEN 'Falabella'
               WHEN 6 THEN 'Transmilenio' WHEN 7 THEN 'Uber Colombia' WHEN 8 THEN 'DiDi Mexico'
               WHEN 9 THEN 'Cinepolis' WHEN 10 THEN 'Netflix CO' WHEN 11 THEN 'Spotify'
               WHEN 12 THEN 'Claro Pagos' WHEN 13 THEN 'ETB' WHEN 14 THEN 'EPM Servicios'
               WHEN 15 THEN 'Bancolombia' WHEN 16 THEN 'Nequi Transfer' WHEN 17 THEN 'Daviplata'
               WHEN 18 THEN 'Jumbo' WHEN 19 THEN 'Carulla' WHEN 20 THEN 'Soriana'
               WHEN 21 THEN 'Liverpool' WHEN 22 THEN 'Mercado Libre CO' WHEN 23 THEN 'Amazon MX'
               WHEN 24 THEN 'Rappi Travel' WHEN 25 THEN 'Avianca' WHEN 26 THEN 'Volaris'
               WHEN 27 THEN 'Bodytech' WHEN 28 THEN 'Smart Fit' WHEN 29 THEN 'Farmatodo'
               WHEN 30 THEN 'Drogueria Olimpica' WHEN 31 THEN 'Homecenter' WHEN 32 THEN 'Coppel'
               WHEN 33 THEN 'Wom' WHEN 34 THEN 'Tigo' WHEN 35 THEN 'Movistar'
               WHEN 36 THEN 'Crepes & Waffles' WHEN 37 THEN 'Frisby' WHEN 38 THEN 'El Corral'
               WHEN 39 THEN 'Dominos Pizza' END,
           CASE MOD(SEQ4(), 6) WHEN 0 THEN 'supermercado' WHEN 1 THEN 'restaurante' WHEN 2 THEN 'transporte'
               WHEN 3 THEN 'entretenimiento' WHEN 4 THEN 'pagos_servicios' WHEN 5 THEN 'transferencia' END,
           CASE MOD(SEQ4(), 10) WHEN 0 THEN 'Bogota' WHEN 1 THEN 'Medellin' WHEN 2 THEN 'Cali'
               WHEN 3 THEN 'Barranquilla' WHEN 4 THEN 'Cartagena' WHEN 5 THEN 'CDMX'
               WHEN 6 THEN 'Guadalajara' WHEN 7 THEN 'Monterrey' WHEN 8 THEN 'Bucaramanga' WHEN 9 THEN 'Pereira' END,
           CASE WHEN MOD(SEQ4(), 10) >= 5 THEN 'MEX' ELSE 'COL' END,
           ROUND(UNIFORM(10, 95, RANDOM())::NUMBER(5,2), 2),
           ROUND(UNIFORM(15000, 800000, RANDOM())::NUMBER(18,2), 2),
           UNIFORM(100, 50000, RANDOM())
    FROM TABLE(GENERATOR(ROWCOUNT => 120))
);

-- Users (600 registros)
INSERT INTO RAPPIPAY_DB.RAW.USERS
SELECT * FROM (
    SELECT 'USR-' || LPAD(SEQ4()::VARCHAR, 5, '0'),
           CASE MOD(SEQ4(), 20) WHEN 0 THEN 'Carlos Martinez' WHEN 1 THEN 'Maria Lopez'
               WHEN 2 THEN 'Juan Hernandez' WHEN 3 THEN 'Laura Garcia' WHEN 4 THEN 'Santiago Ramirez'
               WHEN 5 THEN 'Camila Torres' WHEN 6 THEN 'Andres Morales' WHEN 7 THEN 'Daniela Ruiz'
               WHEN 8 THEN 'Diego Perez' WHEN 9 THEN 'Valentina Castro' WHEN 10 THEN 'Sebastian Diaz'
               WHEN 11 THEN 'Isabella Restrepo' WHEN 12 THEN 'Miguel Rodriguez' WHEN 13 THEN 'Natalia Vargas'
               WHEN 14 THEN 'Alejandro Mendoza' WHEN 15 THEN 'Paula Gutierrez' WHEN 16 THEN 'Fernando Salazar'
               WHEN 17 THEN 'Ana Cardenas' WHEN 18 THEN 'Roberto Aguilar' WHEN 19 THEN 'Monica Rios' END,
           'user_' || SEQ4() || '@rappi.com',
           '+57' || UNIFORM(3001000000, 3209999999, RANDOM())::VARCHAR,
           CASE WHEN MOD(SEQ4(), 5) = 0 THEN 'MEX' ELSE 'COL' END,
           CASE MOD(SEQ4(), 10) WHEN 0 THEN 'Bogota' WHEN 1 THEN 'Medellin' WHEN 2 THEN 'Cali'
               WHEN 3 THEN 'Barranquilla' WHEN 4 THEN 'Cartagena' WHEN 5 THEN 'CDMX'
               WHEN 6 THEN 'Guadalajara' WHEN 7 THEN 'Monterrey' WHEN 8 THEN 'Bucaramanga' WHEN 9 THEN 'Pereira' END,
           DATEADD(DAY, -UNIFORM(30, 1095, RANDOM()), CURRENT_DATE()),
           CASE MOD(SEQ4(), 4) WHEN 0 THEN 'verified' WHEN 1 THEN 'verified' WHEN 2 THEN 'pending' WHEN 3 THEN 'under_review' END,
           UNIFORM(300, 850, RANDOM()),
           CASE WHEN UNIFORM(1,100,RANDOM()) <= 10 THEN 'high' WHEN UNIFORM(1,100,RANDOM()) <= 35 THEN 'medium' ELSE 'low' END,
           UNIFORM(1, 5, RANDOM()), UNIFORM(5, 2000, RANDOM()),
           CASE WHEN UNIFORM(1,100,RANDOM()) <= 5 THEN 'suspended' WHEN UNIFORM(1,100,RANDOM()) <= 10 THEN 'under_review' ELSE 'active' END
    FROM TABLE(GENERATOR(ROWCOUNT => 600))
);

In [ ]:
-- Transactions (12000 registros)
INSERT INTO RAPPIPAY_DB.RAW.TRANSACTIONS
SELECT * FROM (
    SELECT UUID_STRING(),
           'USR-' || LPAD(UNIFORM(0, 599, RANDOM())::VARCHAR, 5, '0'),
           'MCH-' || LPAD(UNIFORM(0, 119, RANDOM())::VARCHAR, 4, '0'),
           ROUND(UNIFORM(5000, 5000000, RANDOM())::NUMBER(18,2), 2),
           CASE WHEN UNIFORM(1,5,RANDOM()) = 1 THEN 'MXN' ELSE 'COP' END,
           CASE MOD(SEQ4(), 6) WHEN 0 THEN 'purchase' WHEN 1 THEN 'transfer' WHEN 2 THEN 'withdrawal'
               WHEN 3 THEN 'payment' WHEN 4 THEN 'refund' WHEN 5 THEN 'top_up' END,
           CASE MOD(SEQ4(), 4) WHEN 0 THEN 'app_mobile' WHEN 1 THEN 'web' WHEN 2 THEN 'pos_terminal' WHEN 3 THEN 'qr_code' END,
           CASE MOD(SEQ4(), 4) WHEN 0 THEN 'android' WHEN 1 THEN 'ios' WHEN 2 THEN 'desktop' WHEN 3 THEN 'tablet' END,
           UNIFORM(10,223,RANDOM())::VARCHAR||'.'||UNIFORM(0,255,RANDOM())::VARCHAR||'.'||UNIFORM(0,255,RANDOM())::VARCHAR||'.'||UNIFORM(1,254,RANDOM())::VARCHAR,
           CASE MOD(SEQ4(), 10) WHEN 0 THEN 'Bogota' WHEN 1 THEN 'Medellin' WHEN 2 THEN 'Cali'
               WHEN 3 THEN 'Barranquilla' WHEN 4 THEN 'Cartagena' WHEN 5 THEN 'CDMX'
               WHEN 6 THEN 'Guadalajara' WHEN 7 THEN 'Monterrey' WHEN 8 THEN 'Bucaramanga' WHEN 9 THEN 'Pereira' END,
           CASE WHEN MOD(SEQ4(), 10) >= 5 THEN 'MEX' ELSE 'COL' END,
           CASE WHEN UNIFORM(1,100,RANDOM()) <= 3 THEN 'declined' WHEN UNIFORM(1,100,RANDOM()) <= 8 THEN 'flagged'
                WHEN UNIFORM(1,100,RANDOM()) <= 12 THEN 'pending_review' ELSE 'approved' END,
           DATEADD(MINUTE, -UNIFORM(1, 525600, RANDOM()), CURRENT_TIMESTAMP()),
           CASE MOD(SEQ4(), 6) WHEN 0 THEN 'Compra en supermercado' WHEN 1 THEN 'Pago transporte'
               WHEN 2 THEN 'Transferencia tercero' WHEN 3 THEN 'Pago servicios publicos'
               WHEN 4 THEN 'Compra restaurante' WHEN 5 THEN 'Recarga celular' END
    FROM TABLE(GENERATOR(ROWCOUNT => 12000))
);

-- Fraud Alerts (250 registros)
INSERT INTO RAPPIPAY_DB.RAW.FRAUD_ALERTS
SELECT * FROM (
    SELECT UUID_STRING(), t.transaction_id, t.user_id,
           CASE MOD(SEQ4(), 5) WHEN 0 THEN 'identity_theft' WHEN 1 THEN 'card_cloning' WHEN 2 THEN 'account_takeover' WHEN 3 THEN 'phishing' WHEN 4 THEN 'money_laundering' END,
           CASE MOD(SEQ4(), 3) WHEN 0 THEN 'critical' WHEN 1 THEN 'high' WHEN 2 THEN 'medium' END,
           CASE MOD(SEQ4(), 4) WHEN 0 THEN 'open' WHEN 1 THEN 'investigating' WHEN 2 THEN 'resolved_fraud' WHEN 3 THEN 'false_positive' END,
           DATEADD(MINUTE, -UNIFORM(1, 43200, RANDOM()), CURRENT_TIMESTAMP()),
           CASE WHEN MOD(SEQ4(), 4) >= 2 THEN DATEADD(MINUTE, -UNIFORM(1, 10080, RANDOM()), CURRENT_TIMESTAMP()) ELSE NULL END,
           CASE MOD(SEQ4(), 3) WHEN 0 THEN 'Fraude confirmado. Tarjeta bloqueada.' WHEN 1 THEN 'Falso positivo confirmado por usuario.' WHEN 2 THEN 'Caso escalado a autoridades.' END,
           CASE MOD(SEQ4(), 4) WHEN 0 THEN 'IP desconocida horario inusual. Monto 4x promedio.'
               WHEN 1 THEN 'Compras rapidas multiples merchants en 5 min.'
               WHEN 2 THEN 'Cambio contrasena + transferencia inmediata.'
               WHEN 3 THEN 'Microtransacciones misma cuenta cada 2 min.' END
    FROM (SELECT transaction_id, user_id, ROW_NUMBER() OVER (ORDER BY RANDOM()) AS rn
          FROM RAPPIPAY_DB.RAW.TRANSACTIONS WHERE status IN ('flagged','declined','pending_review') QUALIFY rn <= 250) t
);

### Paso 1.4: Validar Datos

In [ ]:
SELECT 'TRANSACTIONS' AS tabla, COUNT(*) AS filas FROM RAPPIPAY_DB.RAW.TRANSACTIONS
UNION ALL SELECT 'USERS', COUNT(*) FROM RAPPIPAY_DB.RAW.USERS
UNION ALL SELECT 'MERCHANTS', COUNT(*) FROM RAPPIPAY_DB.RAW.MERCHANTS
UNION ALL SELECT 'FRAUD_ALERTS', COUNT(*) FROM RAPPIPAY_DB.RAW.FRAUD_ALERTS;

### Checklist Task 1
- [ ] RAPPIPAY_DB creada con 4 schemas
- [ ] 12,000 transacciones, 600 usuarios, 120 merchants, 250 alertas
- [ ] Datos realistas (Bogota, Medellin, Oxxo, Nequi, etc.)

---

## Task 2: Explorar Datos con Cortex Code (10 min)

**Objetivo**: Usar Cortex Code para descubrir patrones de fraude.

### Paso 2.1: Analizar modelo de datos

Abre **Cortex Code** y copia este prompt:

```
Conectate a RAPPIPAY_DB y analiza las tablas en el schema RAW.
Dame un resumen: que tablas hay, cuantos registros, como se relacionan,
y que campos son relevantes para deteccion de fraude.
```

### Paso 2.2: Descubrir patrones

```
Usando RAPPIPAY_DB.RAW, analiza patrones de fraude:
1. Tasa de fraude general (flagged+declined vs total)
2. Tipos de fraude mas comunes en FRAUD_ALERTS
3. Merchants con mayor tasa de transacciones sospechosas
4. Correlacion entre monto y probabilidad de fraude
Muestra queries SQL ejecutables.
```

### Paso 2.3: Ejecuta para comparar

In [ ]:
SELECT
    COUNT(*) AS total_transacciones,
    COUNT(CASE WHEN status IN ('flagged', 'declined') THEN 1 END) AS sospechosas,
    ROUND(COUNT(CASE WHEN status IN ('flagged', 'declined') THEN 1 END) * 100.0 / COUNT(*), 2) AS tasa_fraude_pct,
    ROUND(SUM(CASE WHEN status IN ('flagged', 'declined') THEN amount ELSE 0 END), 0) AS monto_en_riesgo
FROM RAPPIPAY_DB.RAW.TRANSACTIONS;

### Checklist Task 2
- [ ] Cortex Code analizo el modelo correctamente
- [ ] Identificaste tasa de fraude y patrones principales

---

## Task 3: Crear Semantic View para Cortex Analyst (10 min)

**Objetivo**: Crear una Semantic View que ensene a la AI tu modelo de datos.

### Paso 3.1: Entender Semantic Views

Pregunta a Cortex Code:
```
Explica que es una Semantic View de Cortex Analyst y por que es importante
para queries en lenguaje natural. Mantenlo breve - 3 oraciones.
```

### Paso 3.2: Generar la Semantic View

```
Crea una Semantic View SQL para RAPPIPAY_DB. Procede autonomamente.
La vista debe:
1. Incluir RAPPIPAY_DB.RAW.TRANSACTIONS, USERS, MERCHANTS y FRAUD_ALERTS
2. Definir relaciones: TRANSACTIONS(USER_ID) -> USERS(USER_ID),
   TRANSACTIONS(MERCHANT_ID) -> MERCHANTS(MERCHANT_ID),
   FRAUD_ALERTS(TRANSACTION_ID) -> TRANSACTIONS(TRANSACTION_ID)
3. Facts: AMOUNT, CREDIT_SCORE, RISK_SCORE
4. Dimensions: TRANSACTION_TYPE, CHANNEL, LOCATION_CITY, LOCATION_COUNTRY,
   CREATED_AT, FULL_NAME, RISK_LEVEL, MERCHANT_NAME, CATEGORY,
   ALERT_TYPE, SEVERITY
5. Ubicacion: RAPPIPAY_DB.ANALYTICS
6. Usa sintaxis: CREATE OR REPLACE SEMANTIC VIEW ... TABLES() RELATIONSHIPS() FACTS() DIMENSIONS()
Ejecuta el SQL.
```

### Paso 3.3: SQL alternativo (si Cortex Code no lo genera correctamente)

In [ ]:
-- Semantic View validada - ejecutar si el prompt no funciono
CREATE OR REPLACE SEMANTIC VIEW RAPPIPAY_DB.ANALYTICS.SV_FRAUD_ANALYTICS
    TABLES (
        RAPPIPAY_DB.RAW.TRANSACTIONS PRIMARY KEY (TRANSACTION_ID)
            COMMENT = 'Transacciones de pagos digitales RappiPay en Colombia y Mexico',
        RAPPIPAY_DB.RAW.USERS PRIMARY KEY (USER_ID)
            COMMENT = 'Usuarios con perfil de riesgo crediticio',
        RAPPIPAY_DB.RAW.MERCHANTS PRIMARY KEY (MERCHANT_ID)
            COMMENT = 'Comercios donde se realizan pagos',
        RAPPIPAY_DB.RAW.FRAUD_ALERTS PRIMARY KEY (ALERT_ID)
            COMMENT = 'Alertas de fraude con tipo severidad y estado'
    )
    RELATIONSHIPS (
        USER_TXN AS TRANSACTIONS(USER_ID) REFERENCES USERS(USER_ID),
        MERCHANT_TXN AS TRANSACTIONS(MERCHANT_ID) REFERENCES MERCHANTS(MERCHANT_ID),
        ALERT_TXN AS FRAUD_ALERTS(TRANSACTION_ID) REFERENCES TRANSACTIONS(TRANSACTION_ID)
    )
    FACTS (
        TRANSACTIONS.AMOUNT AS AMOUNT,
        USERS.CREDIT_SCORE AS CREDIT_SCORE,
        MERCHANTS.RISK_SCORE AS RISK_SCORE
    )
    DIMENSIONS (
        TRANSACTIONS.TRANSACTION_TYPE AS TRANSACTION_TYPE,
        TRANSACTIONS.CHANNEL AS CHANNEL,
        TRANSACTIONS.LOCATION_CITY AS LOCATION_CITY,
        TRANSACTIONS.LOCATION_COUNTRY AS LOCATION_COUNTRY,
        TRANSACTIONS.CREATED_AT AS CREATED_AT,
        TRANSACTIONS.DESCRIPTION AS DESCRIPTION,
        USERS.FULL_NAME AS FULL_NAME,
        USERS.RISK_LEVEL AS RISK_LEVEL,
        USERS.KYC_STATUS AS KYC_STATUS,
        MERCHANTS.MERCHANT_NAME AS MERCHANT_NAME,
        MERCHANTS.CATEGORY AS CATEGORY,
        FRAUD_ALERTS.ALERT_TYPE AS ALERT_TYPE,
        FRAUD_ALERTS.SEVERITY AS SEVERITY,
        FRAUD_ALERTS.RESOLUTION_NOTES AS RESOLUTION_NOTES,
        FRAUD_ALERTS.INVESTIGATOR_NOTES AS INVESTIGATOR_NOTES
    );

In [ ]:
-- Verificar
SHOW SEMANTIC VIEWS IN SCHEMA RAPPIPAY_DB.ANALYTICS;

### Checklist Task 3
- [ ] Semantic View creada en RAPPIPAY_DB.ANALYTICS
- [ ] SHOW SEMANTIC VIEWS muestra SV_FRAUD_ANALYTICS

---

## Task 4: Crear Cortex Agent + Intelligence (10 min)

**Objetivo**: Crear un agente AI que responda preguntas de fraude en espanol.

### Paso 4.1: Entender Snowflake Intelligence

Pregunta a Cortex Code:
```
Que es Snowflake Intelligence y como se diferencia de Cortex Analyst?
Como trabajan juntos? 3 oraciones.
```

### Paso 4.2: Crear el Agent

```
Crea un Agent en RAPPIPAY_DB.APP llamado RAPPIPAY_FRAUD_ANALYST.
Usa la semantic view RAPPIPAY_DB.ANALYTICS.SV_FRAUD_ANALYTICS.
Instrucciones: "Eres un analista senior de fraude de RappiPay. Respondes
sobre transacciones sospechosas, alertas y metricas de riesgo. Incluye
numeros concretos. Responde en espanol."
Preguntas ejemplo: alertas abiertas, tasa fraude por ciudad, merchants sospechosos.
Otorga USAGE a PUBLIC. Ejecuta todo.
```

### Paso 4.3: SQL alternativo

In [ ]:
-- Agent validado - ejecutar si el prompt no funciono
CREATE OR REPLACE AGENT RAPPIPAY_DB.APP.RAPPIPAY_FRAUD_ANALYST
FROM SPECIFICATION $$
{
  "models": {"orchestration": "auto"},
  "instructions": {
    "response": "Responde en espanol de forma concisa. Incluye numeros concretos y periodo de tiempo.",
    "orchestration": "Eres un analista senior de fraude de RappiPay. Ayudas al equipo de riesgo a entender patrones de fraude, alertas activas y metricas de transacciones sospechosas.",
    "sample_questions": [
      {"question": "Cuantas alertas de fraude hay abiertas?"},
      {"question": "Cual es la tasa de fraude por ciudad?"},
      {"question": "Que merchants tienen mas transacciones sospechosas?"},
      {"question": "Cual es el monto total en riesgo?"},
      {"question": "Que tipo de fraude es mas comun?"}
    ]
  },
  "tools": [
    {
      "tool_spec": {
        "type": "cortex_analyst_text_to_sql",
        "name": "analizar_fraude",
        "description": "Consultar metricas de fraude RappiPay: transacciones, alertas, merchants, tasas."
      }
    }
  ],
  "tool_resources": {
    "analizar_fraude": {
      "semantic_view": "RAPPIPAY_DB.ANALYTICS.SV_FRAUD_ANALYTICS"
    }
  }
}
$$;

In [ ]:
-- Grants y verificacion
GRANT USAGE ON DATABASE RAPPIPAY_DB TO ROLE PUBLIC;
GRANT USAGE ON SCHEMA RAPPIPAY_DB.APP TO ROLE PUBLIC;
GRANT USAGE ON AGENT RAPPIPAY_DB.APP.RAPPIPAY_FRAUD_ANALYST TO ROLE PUBLIC;
GRANT SELECT ON ALL SEMANTIC VIEWS IN SCHEMA RAPPIPAY_DB.ANALYTICS TO ROLE PUBLIC;

SHOW AGENTS IN SCHEMA RAPPIPAY_DB.APP;

### Paso 4.4: Probar en Snowflake Intelligence

Ve a **Snowsight > Intelligence** (menu lateral izquierdo) y prueba:

| # | Pregunta | Que valida |
|---|----------|-----------|
| 1 | "Cuantas alertas de fraude hay abiertas?" | Filtro + count |
| 2 | "Cual es la tasa de fraude por ciudad?" | Agrupacion + calculo |
| 3 | "Que merchants tienen mas transacciones sospechosas?" | Top N + join |
| 4 | "Compara fraude entre Colombia y Mexico" | Comparacion |
| 5 | "Dame un resumen del estado del fraude" | Sintesis AI |

### Checklist Task 4
- [ ] Agent RAPPIPAY_FRAUD_ANALYST creado
- [ ] SHOW AGENTS lo muestra
- [ ] Visible en Snowflake Intelligence
- [ ] Responde en espanol con datos correctos

---

## Task 5: Validacion Final (5 min)

### Generar talking points con Cortex Code

```
Basado en RAPPIPAY_DB, genera un script de presentacion de 5 min:
1. Abre con el problema: equipo de fraude tarda horas en reportes adhoc
2. Muestra pregunta en lenguaje natural respondida en segundos
3. Destaca que el SQL es auditable (no caja negra)
4. Cierra con valor: self-service analytics para riesgo
Bullets breves con queries exactas para la demo.
```

In [ ]:
-- Validacion final
SELECT 'Database' AS componente, 'RAPPIPAY_DB' AS nombre
UNION ALL SELECT 'Transacciones', (SELECT COUNT(*)::VARCHAR FROM RAPPIPAY_DB.RAW.TRANSACTIONS)
UNION ALL SELECT 'Alertas', (SELECT COUNT(*)::VARCHAR FROM RAPPIPAY_DB.RAW.FRAUD_ALERTS)
UNION ALL SELECT 'Semantic View', 'SV_FRAUD_ANALYTICS'
UNION ALL SELECT 'Agent', 'RAPPIPAY_FRAUD_ANALYST';

### Cleanup
```sql
-- Descomenta para limpiar:
-- DROP DATABASE IF EXISTS RAPPIPAY_DB;
-- DROP WAREHOUSE IF EXISTS RAPPIPAY_WH;
```

---
## Resumen

| Componente | Producto | Funcion |
|-----------|---------|---------|
| Datos | Tables | 12K transacciones fraude fintech |
| Semantic View | Cortex Analyst | Modelo para lenguaje natural |
| Agent | Intelligence | Chat sobre fraude en espanol |
| Exploracion | Cortex Code | SQL asistido por AI |

### Key Takeaways
- **Cortex Code genera todo** — semantic view y agent sin escribir SQL manualmente
- **Semantic Views son SQL** — no YAML, facil de versionar
- **SQL auditable** — siempre ves que query se ejecuto
- **Prueba con preguntas reales** — usa lo que tu equipo preguntaria

---
*Lab creado por Juan Camilo Villarreal | Snowflake SE LATAM*